# Dịch Dataset chữ Hán → Việt bằng EraX trên L4 offline
Add Input: (1) model từ `download_erax_model.ipynb`, (2) Dataset/Output OCR chứa `.txt` chữ Hán. Không cần repo, wheel bundle tự tạo hoặc HF_TOKEN khi inference.
Notebook dùng BF16 + attention SDPA của Transformers, không dùng FlashAttention và không cần flash-attn. Decode sau khi sinh để hỗ trợ batch. Batch 64 trên mỗi GPU (tổng tối đa 256 đoạn trên 4 GPU); tự chia nhỏ khi CUDA OOM. Không chuyển sang FP16.

In [11]:
import sys
import torch
import importlib.metadata as md

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)

for name in ["transformers", "accelerate"]:
    try:
        print(name, md.version(name))
    except md.PackageNotFoundError:
        print(name, "CHƯA CÀI")

Python: 3.13.16
Torch: 2.11.0+cu130
CUDA: 13.0
transformers 5.20.2
accelerate 1.15.0


In [15]:
from pathlib import Path
import sys, os
MOCK_RUN = False  # True: commit chuẩn bị dependencies; False: chạy tương tác trên L4
MODEL_DIR = '/kaggle/input/notebooks/nthday/erax-model'  # để trống: tự tìm model đã tải trong /kaggle/input
INPUT_DIR = '/kaggle/input/datasets/bamboonguyen/test-ocr/test_dich_zh'  # sửa thành thư mục TXT chữ Hán
OUTPUT_BASE = '/kaggle/working/lan1'
BATCH = 64  # số đoạn MỖI GPU; 4 GPU xử lý tối đa 256 đoạn/lượt
N_GPU = 4
CHUNK_CHARS = 300
MAX_NEW_TOKENS = 1024
MIN_HAN = 50
BOOKS = []  # [] = tất cả; hoặc glob tên file, ví dụ ['*kinh*']
LIMIT = None  # dịch đầy đủ lần 1
RESUME_FROM = []  # các thư mục Output cũ được Add Input; tìm dich_raw/*.jsonl

Chế độ: INFERENCE — yêu cầu L4 và dependencies đã cài.


In [16]:
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
import json, time, importlib.metadata
try:
    import torch, transformers
except ImportError as exc:
    raise RuntimeError('Thiếu dependencies. Chuẩn bị Dependency Manager khi online, rồi chạy session mới.') from exc
from transformers import AutoTokenizer, AutoProcessor, Gemma3ForConditionalGeneration
assert torch.cuda.is_available(), 'CUDA không khả dụng: kiểm tra driver/GPU.'
assert torch.cuda.is_bf16_supported(), 'GPU phải hỗ trợ BF16 trực tiếp; không dùng T4.'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('Transformers:', transformers.__version__, 'Attention: sdpa')
versions = {name: importlib.metadata.version(name) for name in ('torch', 'transformers', 'accelerate', 'huggingface-hub')}
print('Dependency versions:', versions)
direct_url = importlib.metadata.distribution('transformers').read_text('direct_url.json')
if direct_url:
    print('Transformers source:', direct_url)
if not MODEL_DIR:
    candidates = list(Path('/kaggle/input').rglob('erax-model-manifest.json')) if Path('/kaggle/input').exists() else []
    if len(candidates) != 1:
        raise RuntimeError('Điền MODEL_DIR: thư mục chứa config.json và các file safetensors của EraX.')
    MODEL_DIR = str(candidates[0].parent)
# MODEL_DIR có thể là Output gốc hoặc thư mục model bên trong.
model_root = Path(MODEL_DIR)
if not model_root.is_dir():
    raise RuntimeError(f'MODEL_DIR không tồn tại: {model_root}. Kiểm tra Add Input và đường dẫn.')
if not (model_root / 'config.json').is_file():
    candidates = sorted({p.parent for p in model_root.rglob('config.json')
                         if (p.parent / 'tokenizer_config.json').is_file()})
    if len(candidates) != 1:
        raise RuntimeError(f'Tìm thấy {len(candidates)} thư mục model trong {model_root}: {candidates}. '
                           'Điền MODEL_DIR trực tiếp tới thư mục chứa config.json và tokenizer.')
    model_root = candidates[0]
print('Model directory:', model_root)
print('Tokenizer files:', sorted(p.name for p in model_root.glob('*token*')))
if not (model_root / 'tokenizer.json').is_file():
    raise RuntimeError(f'Thiếu tokenizer.json trong {model_root}. '
                       'Chạy lại download_erax_model.ipynb khi Internet On để tải đủ model/tokenizer, '
                       'Save Version rồi Add Input Output mới. Không thể tải bổ sung trong session offline.')
model_path = str(model_root)
model_manifest_path = model_root / 'erax-model-manifest.json'
if model_manifest_path.exists():
    model_manifest = json.loads(model_manifest_path.read_text())
    print('Model commit:', model_manifest['revision'])
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True, padding_side='left')
processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)
assert torch.cuda.device_count() >= N_GPU, f'Cần {N_GPU} GPU, chỉ thấy {torch.cuda.device_count()}'
assert all(torch.cuda.get_device_capability(i)[0] >= 8 for i in range(N_GPU)), 'BF16 yêu cầu GPU Ampere trở lên; không dùng T4.'
from concurrent.futures import ThreadPoolExecutor
# Mỗi GPU một replica; không dùng device_map="auto" (chỉ chia model, không chia batch).
models = []
for gpu in range(N_GPU):
    with torch.cuda.device(gpu):
        replica = Gemma3ForConditionalGeneration.from_pretrained(
            model_path, local_files_only=True, dtype=torch.bfloat16,
            attn_implementation='sdpa', device_map={'': gpu},
        ).eval()
        models.append(replica)
        print('GPU', gpu, torch.cuda.get_device_name(gpu), replica.dtype, replica.config._attn_implementation)
model = models[0]  # smoke test trên GPU 0; dịch dataset dùng cả 4 replica


GPU: NVIDIA L4 Torch: 2.11.0+cu130 CUDA: 13.0
Transformers: 5.20.2 Attention: sdpa
Dependency versions: {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'accelerate': '1.15.0', 'huggingface-hub': '1.33.0'}
Model directory: /kaggle/input/notebooks/nthday/erax-model/EraX-Translator-V1.0
Tokenizer files: ['added_tokens.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json']
Model commit: f28cb898746c9396a11497f8008312309ae7dc85


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

GPU 0 NVIDIA L4 torch.bfloat16 sdpa


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

GPU 1 NVIDIA L4 torch.bfloat16 sdpa


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

GPU 2 NVIDIA L4 torch.bfloat16 sdpa


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

GPU 3 NVIDIA L4 torch.bfloat16 sdpa


In [17]:

SYSTEM = '''Bạn là Trợ lý AI xuất sắc về dịch thuật nhiều ngôn ngữ, đặc biệt tiếng Anh, tiếng Trung Hoa, tiếng Việt.
Bạn cũng là 1 Hoà thượng Phật giáo uyên thâm về dịch thuật Cổ văn Trung Quốc. Người dùng sẽ giao nhiệm vụ dịch thuật cho bạn từ ngôn ngữ bất kỳ sang một ngôn ngữ được chỉ định.
Nhiệm vụ của bạn là dịch thật sát nghĩa, thể hiện đúng ý của bài gốc và không chế tác hay bịa đặt gì thêm. Đặc biệt lưu ý danh xưng phải giữ nguyên vẹn, dịch đúng tên người, tên địa danh phải tuyệt đối chính xác. Không được bình luận, không được cung cấp lời giới thiệu hay mở bài hay kết luận gì, chỉ dịch thật sát nghĩa và không bỏ qua bất kỳ ý hay từ nào.
'''
TEXTS = ['三寶者，吾輩塵世之至尊也。夫欲出家者，始亦皈依三寶，繼受五戒，乃至八關齋戒，其後方成沙彌。此誠佛道常軌，凡奉佛國，咸所遵行。吾法華道場亦然。故發宏願：願世世生生，值遇三寶，恭敬供養，依佛聖教，奉行眾善。此即吾等所嚮之鵠的也。']
# Smoke test một đoạn trước khi chạy Dataset. Batch thật lấy từ ô cấu hình.
MAX_LENGTH = 8192  # gồm input + output, như code mẫu HF
results = []
torch.manual_seed(42)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
for offset in range(0, len(TEXTS), 1):
    texts = TEXTS[offset:offset + 1]
    prompts = [tokenizer.apply_chat_template([{'role': 'system', 'content': SYSTEM},
               {'role': 'user', 'content': text + '\nDịch sang tiếng Việt.'}],
               tokenize=False, add_generation_prompt=True) for text in texts]
    # Tránh thêm BOS lần hai vì template đã chèn special tokens.
    enc = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
    assert enc['input_ids'].shape[1] < MAX_LENGTH
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        diagnostic = model(**enc, logits_to_keep=1, use_cache=False)
        finite = torch.isfinite(diagnostic.logits[:, -1, :]).all().item()
        del diagnostic
        assert finite, 'Logits có NaN/Inf: dừng trước khi ghi bản dịch lỗi.'
        gen = model.generate(**enc, max_length=MAX_LENGTH, temperature=0.2, top_k=64,
                             top_p=0.95, min_p=0.0, repetition_penalty=1.05,
                             do_sample=True, pad_token_id=tokenizer.pad_token_id,
                             remove_invalid_values=False)
    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    for text, ids in zip(texts, gen[:, enc['input_ids'].shape[1]:].cpu()):
        vi = tokenizer.decode(ids, skip_special_tokens=True).strip()
        raw = tokenizer.decode(ids, skip_special_tokens=False)
        assert vi, f'Bản dịch rỗng, raw={raw!r}'
        results.append(dict(zh=text, vi=vi, raw=raw, seconds=seconds, hit_max=gen.shape[1] >= MAX_LENGTH))
        print('ZH:', text, '\nVI:', vi, '\nBatch seconds:', round(seconds, 2))
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
output = output_dir / 'erax_l4_offline_results.json'
output.write_text(json.dumps({'versions': versions, 'results': results}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Output:', output)


ZH: 三寶者，吾輩塵世之至尊也。夫欲出家者，始亦皈依三寶，繼受五戒，乃至八關齋戒，其後方成沙彌。此誠佛道常軌，凡奉佛國，咸所遵行。吾法華道場亦然。故發宏願：願世世生生，值遇三寶，恭敬供養，依佛聖教，奉行眾善。此即吾等所嚮之鵠的也。 
VI: Tam Bảo là đấng tối tôn của chúng ta trong cõi trần gian này. Đối với những người xuất gia, trước hết họ phải quy y Tam Bảo, tiếp đó là thọ ngũ giới, rồi bát quan trai giới, sau đó mới trở thành Sa Di. Đây quả thật là con đường tu hành Phật đạo thông thường, mà tất cả các nước theo Phật đều làm theo. Đạo Tràng Pháp Hoa của chúng ta cũng vậy. Vì thế, chúng ta phát nguyện: Phát nguyện đời đời kiếp kiếp gặp được Tam Bảo, cung kính cúng dường, vâng làm theo lời dạy của Phật và các vị Thánh, vâng làm theo mọi điều thiện. Đây chính là mục tiêu mà chúng ta hướng đến. 
Batch seconds: 10.81
Output: /kaggle/working/erax_l4_offline_results.json


## Dịch toàn bộ Dataset
Mỗi file TXT đầu vào → một file UTF-8 `runs/dich_erax_l4_<hash>/dich/<tên sách>.txt`. Các đoạn cách nhau một dòng trống; chưa dịch ghi `[chưa dịch]`. Raw JSONL được flush sau mỗi đoạn hợp lệ. Resume chỉ dùng bản dịch không rỗng và có cùng cấu hình/model revision. Các đoạn chạm giới hạn được đánh dấu `hit_max` để kiểm tra lại. Không tự bảo đảm thuật ngữ nhất quán: cần đánh giá bản dịch thực tế.

In [18]:
import re, hashlib, fnmatch
from tqdm.auto import tqdm
source_dir = Path(INPUT_DIR)
assert source_dir.is_dir(), 'Sửa INPUT_DIR thành thư mục TXT của Dataset OCR.'
assert BATCH > 0 and CHUNK_CHARS > 0 and MAX_NEW_TOKENS > 0
assert LIMIT is None or LIMIT > 0
han = re.compile(r'[\u3400-\u4dbf\u4e00-\u9fff\U00020000-\U0002ffff]')
identity = json.loads(model_manifest_path.read_text())['revision'] if model_manifest_path.exists() else str(Path(model_path).resolve())
settings = dict(model=identity, system=SYSTEM, instruction='Dịch sang tiếng Việt.', chunk_chars=CHUNK_CHARS,
                max_new_tokens=MAX_NEW_TOKENS, temperature=0.2, top_k=64, top_p=0.95, min_p=0.0, repetition_penalty=1.05,
                dtype='bfloat16', attention='sdpa', versions=versions, min_han=MIN_HAN, books=BOOKS)
config_id = hashlib.sha256(json.dumps(settings, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:12]
run_dir = Path(OUTPUT_BASE) / 'runs' / ('dich_erax_l4_' + config_id)
raw_dir, translated_dir = run_dir / 'dich_raw', run_dir / 'dich'
raw_dir.mkdir(parents=True, exist_ok=True)
translated_dir.mkdir(exist_ok=True)
(run_dir / 'run_manifest.json').write_text(json.dumps(dict(config=settings, input_dir=INPUT_DIR, batch_per_gpu=BATCH, n_gpu=N_GPU, global_batch=BATCH * N_GPU, limit=LIMIT), ensure_ascii=False, indent=2), encoding='utf-8')
books, chunks = {}, []
for path in sorted(source_dir.rglob('*.txt')):
    if BOOKS and not any(fnmatch.fnmatch(path.stem, pattern) for pattern in BOOKS):
        continue
    text = path.read_text(encoding='utf-8-sig')
    if len(han.findall(text)) < MIN_HAN:
        continue
    # Phân tách theo câu/xuống dòng, nhưng không bỏ chữ hoặc cắt bỏ phần dài.
    parts, current = [], ''
    for piece in re.split(r'(?<=[。！？；\n])', text):
        while piece:
            take = min(CHUNK_CHARS - len(current), len(piece))
            current += piece[:take]
            piece = piece[take:]
            if len(current) == CHUNK_CHARS:
                if current.strip(): parts.append(current.strip())
                current = ''
        if len(current) >= CHUNK_CHARS // 2:
            if current.strip(): parts.append(current.strip())
            current = ''
    if current.strip(): parts.append(current.strip())
    relative = path.relative_to(source_dir)
    name = str(relative.with_suffix('')).replace('/', '__')
    assert name not in books, f'Trùng tên output: {name}'
    records = []
    for index, zh in enumerate(parts):
        key = hashlib.sha256((config_id + str(relative) + str(index) + zh).encode()).hexdigest()
        records.append(dict(id=key, book=name, index=index, zh=zh))
    books[name] = records
    chunks.extend(records)
assert chunks, 'Không có TXT đủ chữ Hán phù hợp INPUT_DIR/BOOKS/MIN_HAN.'
done = {}
for folder in [*map(Path, RESUME_FROM), run_dir]:
    for path in folder.rglob('dich_raw/*.jsonl'):
        with path.open(encoding='utf-8') as stream:
            for line in stream:
                try:
                    row = json.loads(line)
                    if row.get('config_id') == config_id and row.get('vi', '').strip():
                        done[row['id']] = row
                except (ValueError, KeyError):
                    continue
wanted = {c['id'] for c in chunks}
done = {key: row for key, row in done.items() if key in wanted}
todo = [c for c in chunks if c['id'] not in done]
if LIMIT is not None: todo = todo[:LIMIT]
print('Run:', run_dir, '| Sách:', len(books), '| Tổng đoạn:', len(chunks), '| Resume:', len(done), '| Chọn:', len(todo))
# Lưu cả dữ liệu resume vào run hiện tại để Output mới có thể resume độc lập.
with (raw_dir / 'resume.jsonl').open('w', encoding='utf-8') as stream:
    for row in done.values(): stream.write(json.dumps(row, ensure_ascii=False) + '\n')
def export_books():
    for name, records in books.items():
        text = '\n\n'.join(done[c['id']]['vi'] if c['id'] in done else '[chưa dịch]' for c in records)
        (translated_dir / (name + '.txt')).write_text(text + '\n', encoding='utf-8')
def translate_batch(gpu, batch):
    if not batch:
        return []
    device = f'cuda:{gpu}'
    with torch.cuda.device(gpu):
        prompts = [tokenizer.apply_chat_template(
            [{'role': 'system', 'content': SYSTEM},
             {'role': 'user', 'content': c['zh'] + '\nDịch sang tiếng Việt.'}],
            tokenize=False, add_generation_prompt=True) for c in batch]
        enc = tokenizer(prompts, return_tensors='pt', padding=True,
                        add_special_tokens=False).to(device)
        try:
            with torch.inference_mode():
                gen = models[gpu].generate(
                    **enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                    temperature=0.2, top_k=64, top_p=0.95, min_p=0.0,
                    repetition_penalty=1.05, pad_token_id=tokenizer.pad_token_id,
                    remove_invalid_values=False)
            generated = gen[:, enc['input_ids'].shape[1]:].cpu().tolist()
        except torch.cuda.OutOfMemoryError:
            del enc
            if len(batch) == 1:
                raise
            torch.cuda.empty_cache()
            middle = len(batch) // 2
            print(f'GPU {gpu}: OOM batch {len(batch)}, chia nhỏ để thử lại', flush=True)
            return translate_batch(gpu, batch[:middle]) + translate_batch(gpu, batch[middle:])
        eos = models[gpu].generation_config.eos_token_id
        eos = set(eos if isinstance(eos, list) else [eos])
        rows = []
        for record, ids in zip(batch, generated):
            vi = tokenizer.decode(ids, skip_special_tokens=True).strip()
            if not vi:
                raise RuntimeError(f'Bản dịch rỗng: {record["id"]}')
            rows.append(dict(record, config_id=config_id, vi=vi,
                             hit_max=len(ids) >= MAX_NEW_TOKENS and not any(t in eos for t in ids)))
        return rows

try:
    with ThreadPoolExecutor(max_workers=N_GPU) as pool, \
         (raw_dir / 'translations.jsonl').open('a', encoding='utf-8') as stream, \
         tqdm(total=len(todo), desc='Dịch EraX 4 GPU', unit='đoạn') as bar:
        for start in range(0, len(todo), BATCH * N_GPU):
            batch = todo[start:start + BATCH * N_GPU]
            # Mỗi GPU nhận tối đa BATCH đoạn; 64/GPU -> tối đa 256/lượt.
            futures = [pool.submit(translate_batch, gpu,
                                   batch[gpu * BATCH:(gpu + 1) * BATCH])
                       for gpu in range(N_GPU)]
            for future in futures:
                for row in future.result():
                    stream.write(json.dumps(row, ensure_ascii=False) + '\n')
                    stream.flush()
                    done[row['id']] = row
                    bar.update(1)
finally:
    export_books()
print('Đã dịch:', len(done), '/', len(chunks), '| Chạm max:', sum(r.get('hit_max', False) for r in done.values()))
print('Output:', translated_dir)


Run: /kaggle/working/lan1/runs/dich_erax_l4_6264fcf22ac7 | Sách: 165 | Tổng đoạn: 166 | Resume: 0 | Chọn: 166


Dịch EraX 4 GPU:   0%|          | 0/166 [00:00<?, ?đoạn/s]

Đã dịch: 166 / 166 | Chạm max: 0
Output: /kaggle/working/lan1/runs/dich_erax_l4_6264fcf22ac7/dich


In [20]:
from pathlib import Path
import shutil
from IPython.display import FileLink, display

folder = Path("/kaggle/working/lan1/runs/dich_erax_l4_6264fcf22ac7/dich")
assert folder.is_dir(), f"Không tìm thấy: {folder}"

zip_path = shutil.make_archive(
    "/kaggle/working/test_dich",
    "zip",
    root_dir=folder,
)
display(FileLink(zip_path))


/kaggle/working/test_dich.zip